In [59]:
#all necessary imports
import numpy as np
import pandas as pd
import polars as pl
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# TA indicators
import ta
from ta.trend import PSARIndicator

# Data Normalization and Scaling
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# Scikit-learn utilities
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# hyperparameter optimization
import optuna
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Training device:", device)

print("Torch version:", torch.__version__)
print("Polars version:", pl.__version__)
print("TQDM imported successfully!")
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)
print("TA library imported successfully!")
print("Optuna version:",optuna.__version__)

Training device: cuda
Torch version: 2.5.1+cu121
Polars version: 0.20.26
TQDM imported successfully!
NumPy version: 1.26.4
Pandas version: 2.2.2
TA library imported successfully!
Optuna version: 4.8.0


In [60]:
df = pd.read_csv("2110_daily.csv")
print(df.shape)
print(df.head())
print(df.dtypes)
print(df.isnull().sum())

(4719, 18)
              datetime  ticker  open_price  high_price   low_price  \
0  2004-02-24 16:00:00    2110  263.966867  266.688175  263.966867   
1  2004-02-25 16:00:00    2110  263.966867  277.573406  263.059764   
2  2004-02-28 16:00:00    2110  273.944996  282.108920  272.130791   
3  2004-02-29 16:00:00    2110  281.201817  307.507793  277.573406   
4  2004-03-01 16:00:00    2110  308.414896  323.835641  299.343870   

   close_price        volume      EMA_10  MACD_hist     RSI_14      ROC_5  \
0   264.873970  4.147299e+04  265.771898  -0.072458  56.233094  -1.016949   
1   276.666304  8.761780e+05  267.752699   0.419638  67.094818   3.741497   
2   280.294714  5.455675e+05  270.033066   0.895806  69.595147   2.657807   
3   307.507793  1.519195e+06  276.846652   2.840108  81.158673  14.141414   
4   299.343870  1.450159e+06  280.937056   3.338532  72.277753  13.013699   

     ATR_14    BB_pct           OBV  volume_ratio     MFI_14  \
0  6.410805  0.637412  2.771299e+06      

In [61]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import pickle

# ── 1. Load ───────────────────────────────────────────────────────────────────
df      = pd.read_csv("2110_daily.csv", parse_dates=["datetime"])
tasi_df = pd.read_csv("tasi_daily.csv", parse_dates=["datetime"])

df      = df.sort_values("datetime").reset_index(drop=True)
tasi_df = tasi_df.sort_values("datetime").reset_index(drop=True)

# ── 2. Prepare TASI features ──────────────────────────────────────────────────
tasi_df["TASI_return"]       = tasi_df["close_price"].pct_change(1) * 100
tasi_df["TASI_volume_ratio"] = tasi_df["volume"] / tasi_df["volume"].rolling(20).mean()

tasi_merge = tasi_df[["datetime", "TASI_return", "TASI_volume_ratio"]].dropna()

# ── 3. Merge TASI into 2110 ───────────────────────────────────────────────────
df = pd.merge(df, tasi_merge, on="datetime", how="inner")
print(f"Rows after TASI merge: {len(df):,}")

# ── 4. Add lag features ───────────────────────────────────────────────────────
for lag in range(1, 6):
    df[f"close_lag_{lag}"]   = df["close_price"].shift(lag)
    df[f"returns_lag_{lag}"] = df["close_price"].pct_change(1).shift(lag) * 100
    df[f"volume_lag_{lag}"]  = df["volume"].shift(lag)

# ── 5. Fix volume target — log return ─────────────────────────────────────────
df["volume_log"]            = np.log1p(df["volume"])
df["volume_pct_change_log"] = df["volume_log"].diff(1).shift(-1)
df = df.drop(columns=["volume_log", "volume_pct_change"])

df = df.dropna().reset_index(drop=True)
print(f"Rows after lag + dropna: {len(df):,}")

# ── 6. Define columns ─────────────────────────────────────────────────────────
feature_cols = [
    # OHLCV
    'open_price', 'high_price', 'low_price', 'close_price', 'volume',
    # Trend
    'EMA_10', 'MACD_hist',
    # Momentum
    'RSI_14', 'ROC_5',
    # Volatility
    'ATR_14', 'BB_pct',
    # Volume indicators
    'OBV', 'volume_ratio', 'MFI_14',
    # TASI
    'TASI_return', 'TASI_volume_ratio',
    # Lag features
    'close_lag_1', 'close_lag_2', 'close_lag_3', 'close_lag_4', 'close_lag_5',
    'returns_lag_1', 'returns_lag_2', 'returns_lag_3', 'returns_lag_4', 'returns_lag_5',
    'volume_lag_1', 'volume_lag_2', 'volume_lag_3', 'volume_lag_4', 'volume_lag_5',
]

target_cols = ['price_pct_change', 'volume_pct_change_log']

print(f"Total features: {len(feature_cols)}")

# ── 7. Outlier removal ────────────────────────────────────────────────────────
outlier_sigma = {'price_pct_change': 3, 'volume_pct_change_log': 2}

for col, sigma in outlier_sigma.items():
    mean, std = df[col].mean(), df[col].std()
    df        = df[df[col].between(mean - sigma * std, mean + sigma * std)]

df = df.reset_index(drop=True)
print(f"Rows after outlier removal: {len(df):,}")

# ── 8. Train / Val / Test
n         = len(df)
train_end = int(n * 0.75)  
val_end   = int(n * 0.90)   

train_df  = df.iloc[:train_end]
val_df    = df.iloc[train_end:val_end]
test_df   = df.iloc[val_end:]

print(f"Train : {len(train_df):,} rows  ({train_df['datetime'].min().date()} → {train_df['datetime'].max().date()})")
print(f"Val   : {len(val_df):,} rows  ({val_df['datetime'].min().date()} → {val_df['datetime'].max().date()})")
print(f"Test  : {len(test_df):,} rows  ({test_df['datetime'].min().date()} → {test_df['datetime'].max().date()})")

# ── 9. Scale ──────────────────────────────────────────────────────────────────
feature_scaler = StandardScaler()
target_scaler  = StandardScaler()

train_X = feature_scaler.fit_transform(train_df[feature_cols])
val_X   = feature_scaler.transform(val_df[feature_cols])
test_X  = feature_scaler.transform(test_df[feature_cols])

train_y = target_scaler.fit_transform(train_df[target_cols])
val_y   = target_scaler.transform(val_df[target_cols])
test_y  = target_scaler.transform(test_df[target_cols])

# ── 10. Sliding window ────────────────────────────────────────────────────────
SEQ_LEN = 30

def create_sequences(X, y, seq_len):
    sequences, targets = [], []
    for i in range(seq_len, len(X)):
        sequences.append(X[i - seq_len:i])
        targets.append(y[i])
    return np.array(sequences), np.array(targets)

X_train, y_train = create_sequences(train_X, train_y, SEQ_LEN)
X_val,   y_val   = create_sequences(val_X,   val_y,   SEQ_LEN)
X_test,  y_test  = create_sequences(test_X,  test_y,  SEQ_LEN)

print(f"\nX_train : {X_train.shape}  → (samples, seq_len, features)")
print(f"X_val   : {X_val.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape}  → (samples, 2)")

# ── 11. Save scalers ──────────────────────────────────────────────────────────
with open("2110_feature_scaler.pkl", "wb") as f:
    pickle.dump(feature_scaler, f)

with open("2110_target_scaler.pkl", "wb") as f:
    pickle.dump(target_scaler, f)

print("\nScalers saved.")

Rows after TASI merge: 4,719
Rows after lag + dropna: 4,712
Total features: 31
Rows after outlier removal: 4,338
Train : 3,253 rows  (2004-03-03 → 2019-09-12)
Val   : 651 rows  (2019-09-15 → 2024-06-13)
Test  : 434 rows  (2024-06-23 → 2026-05-05)

X_train : (3223, 30, 31)  → (samples, seq_len, features)
X_val   : (621, 30, 31)
X_test  : (404, 30, 31)
y_train : (3223, 2)  → (samples, 2)

Scalers saved.


In [62]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np


# ── Dataset ───────────────────────────────────────────────────────────────────
class TadawulDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# ── DataLoaders ───────────────────────────────────────────────────────────────
BATCH_SIZE = 32

train_dataset = TadawulDataset(X_train, y_train)
val_dataset   = TadawulDataset(X_val,   y_val)
test_dataset  = TadawulDataset(X_test,  y_test)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader   = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")

# ── Sanity check ──────────────────────────────────────────────────────────────
sample_X, sample_y = next(iter(train_loader))
print(f"\nSample batch X : {sample_X.shape}  → (batch, seq_len, features)")
print(f"Sample batch y : {sample_y.shape}  → (batch, 2)")

Train batches : 101
Val batches   : 20
Test batches  : 13

Sample batch X : torch.Size([32, 30, 31])  → (batch, seq_len, features)
Sample batch y : torch.Size([32, 2])  → (batch, 2)


In [63]:
import torch
import torch.nn as nn


class StockPctChangeLSTM(nn.Module):
    def __init__(
        self,
        input_size=31,       
        hidden_size=128,
        num_layers=2,
        dropout=0.2
    ):
        super(StockPctChangeLSTM, self).__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.bn      = nn.BatchNorm1d(hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_size, 2)  # [price_pct_change, volume_pct_change]

    def forward(self, x):
        out, _  = self.lstm(x)
        out     = out[:, -1, :]     # last timestep (batch, hidden_size)
        out     = self.bn(out)
        out     = self.dropout(out)
        out     = self.fc(out)      # (batch, 2)
        return out


# Loss & Optimizer
# criterion = nn.HuberLoss()
# optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

In [64]:
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import ReduceLROnPlateau
import numpy as np
import pickle
import time

# ── Device ────────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ── Model ─────────────────────────────────────────────────────────────────────
model = StockPctChangeLSTM(
    input_size=31,
    hidden_size=64,
    num_layers=2,
    dropout=0.2
).to(device)

# ── Loss & Optimizer ──────────────────────────────────────────────────────────
criterion = nn.HuberLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=5,
                               factor=0.5, verbose=True)

# ── Config ────────────────────────────────────────────────────────────────────
EPOCHS        = 100
PATIENCE      = 15          # early stopping
best_val_loss = float('inf')
patience_ctr  = 0
history       = {"train_loss": [], "val_loss": []}

# ── Training Loop ─────────────────────────────────────────────────────────────
print(f"\n{'Epoch':>6} | {'Train Loss':>10} | {'Val Loss':>10} | {'Time':>6}")
print("-" * 42)

for epoch in range(1, EPOCHS + 1):
    start = time.time()

    # ── Train ─────────────────────────────────────────────────────────────────
    model.train()
    train_losses = []

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        preds = model(X_batch)
        loss  = criterion(preds, y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # gradient clipping
        optimizer.step()
        train_losses.append(loss.item())

    # ── Validate ──────────────────────────────────────────────────────────────
    model.eval()
    val_losses = []

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            preds   = model(X_batch)
            loss    = criterion(preds, y_batch)
            val_losses.append(loss.item())

    train_loss = np.mean(train_losses)
    val_loss   = np.mean(val_losses)
    elapsed    = time.time() - start

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    print(f"{epoch:>6} | {train_loss:>10.6f} | {val_loss:>10.6f} | {elapsed:>5.1f}s")

    # ── Scheduler ─────────────────────────────────────────────────────────────
    scheduler.step(val_loss)

    # ── Early Stopping + Save Best ────────────────────────────────────────────
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_ctr  = 0
        torch.save(model.state_dict(), "2110_best_model.pt")
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch} — no improvement for {PATIENCE} epochs.")
            break

print(f"\nBest val loss: {best_val_loss:.6f}")

Using device: cuda

 Epoch | Train Loss |   Val Loss |   Time
------------------------------------------


c:\Users\User\miniconda3\envs\quant_dl\lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


     1 |   0.435685 |   0.412637 |   0.6s
     2 |   0.414348 |   0.414965 |   0.5s
     3 |   0.405384 |   0.406871 |   0.4s
     4 |   0.403786 |   0.410192 |   0.5s
     5 |   0.400506 |   0.403697 |   0.5s
     6 |   0.398780 |   0.404716 |   0.4s
     7 |   0.398277 |   0.401812 |   0.5s
     8 |   0.395566 |   0.400581 |   0.4s
     9 |   0.393459 |   0.404714 |   0.4s
    10 |   0.390262 |   0.413172 |   0.4s
    11 |   0.391852 |   0.406073 |   0.4s
    12 |   0.390595 |   0.409106 |   0.4s
    13 |   0.389215 |   0.410788 |   0.4s
    14 |   0.386419 |   0.408834 |   0.4s
    15 |   0.383190 |   0.408511 |   0.4s
    16 |   0.380192 |   0.418638 |   0.4s
    17 |   0.377991 |   0.421153 |   0.4s
    18 |   0.376869 |   0.416656 |   0.4s
    19 |   0.374598 |   0.423424 |   0.5s
    20 |   0.371576 |   0.428061 |   0.5s
    21 |   0.368468 |   0.430435 |   0.4s
    22 |   0.368497 |   0.428104 |   0.4s
    23 |   0.369185 |   0.425142 |   0.4s

Early stopping at epoch 23 — no i

In [65]:
# ── Evaluation on Test Set ────────────────────────────────────────────────────
print("\n" + "=" * 50)
print("EVALUATION ON TEST SET")
print("=" * 50)

# Load best model
model.load_state_dict(torch.load("2110_best_model.pt"))
model.eval()

# Load target scaler for inverse transform
with open("2110_target_scaler.pkl", "rb") as f:
    target_scaler = pickle.load(f)

all_preds   = []
all_actuals = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        preds   = model(X_batch).cpu().numpy()
        actuals = y_batch.numpy()
        all_preds.append(preds)
        all_actuals.append(actuals)

all_preds   = np.vstack(all_preds)     # (n_test, 2) — scaled
all_actuals = np.vstack(all_actuals)   # (n_test, 2) — scaled

# Inverse transform back to original scale
all_preds   = target_scaler.inverse_transform(all_preds)
all_actuals = target_scaler.inverse_transform(all_actuals)

# Column 0 → price_pct_change        (already in %, no conversion needed)
# Column 1 → volume_pct_change_log   (log return → convert back to %)
price_preds_pct    = all_preds[:, 0]
price_actuals_pct  = all_actuals[:, 0]

volume_preds_pct   = (np.expm1(all_preds[:, 1]))   * 100
volume_actuals_pct = (np.expm1(all_actuals[:, 1])) * 100

# ── Metrics ───────────────────────────────────────────────────────────────────
def evaluate(actuals, preds, label):
    mae     = np.mean(np.abs(actuals - preds))
    rmse    = np.sqrt(np.mean((actuals - preds) ** 2))
    dir_acc = np.mean(np.sign(actuals) == np.sign(preds)) * 100

    print(f"\n  [{label}]")
    print(f"  MAE               : {mae:.4f}%")
    print(f"  RMSE              : {rmse:.4f}%")
    print(f"  Directional Acc   : {dir_acc:.2f}%")

evaluate(price_actuals_pct,  price_preds_pct,  "Price % Change")
evaluate(volume_actuals_pct, volume_preds_pct, "Volume % Change")

# ── Save predictions for review ───────────────────────────────────────────────
results_df = pd.DataFrame({
    "actual_price_pct"    : price_actuals_pct,
    "predicted_price_pct" : price_preds_pct,
    "actual_volume_pct"   : volume_actuals_pct,
    "predicted_volume_pct": volume_preds_pct,
})
results_df.to_csv("2110_test_predictions.csv", index=False)
print("\nPredictions saved: 2110_test_predictions.csv")


EVALUATION ON TEST SET

  [Price % Change]
  MAE               : 1.5453%
  RMSE              : 2.1245%
  Directional Acc   : 53.96%

  [Volume % Change]
  MAE               : 55.3255%
  RMSE              : 77.2027%
  Directional Acc   : 55.69%

Predictions saved: 2110_test_predictions.csv


C:\Users\User\AppData\Local\Temp\ipykernel_26672\1079301384.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("2110_best_model.pt"))


DIAGNOSTIC

In [66]:
import pandas as pd
import numpy as np

results = pd.read_csv("2110_test_predictions.csv")

price_actual    = results["actual_price_pct"]
price_predicted = results["predicted_price_pct"]

correct   = (np.sign(price_actual) == np.sign(price_predicted))
up_mask   = price_actual > 0
down_mask = price_actual < 0

print("=== Prediction Magnitude ===")
print(f"Actual    mean: {price_actual.mean():.4f}   std: {price_actual.std():.4f}")
print(f"Predicted mean: {price_predicted.mean():.4f}   std: {price_predicted.std():.4f}")

print(f"\n=== Directional Breakdown ===")
print(f"Overall Dir Acc : {correct.mean()*100:.2f}%")
print(f"When actual UP  : {correct[up_mask].mean()*100:.2f}%  ({up_mask.sum()} samples)")
print(f"When actual DOWN: {correct[down_mask].mean()*100:.2f}%  ({down_mask.sum()} samples)")

print(f"\n=== Prediction Bias ===")
print(f"% predicted UP  : {(price_predicted > 0).mean()*100:.2f}%")
print(f"% actual UP     : {(price_actual > 0).mean()*100:.2f}%")

print("=== Mean price_pct_change by split ===")
print(f"Train mean : {train_df['price_pct_change'].mean():.4f}%")
print(f"Val mean   : {val_df['price_pct_change'].mean():.4f}%")
print(f"Test mean  : {test_df['price_pct_change'].mean():.4f}%")
print(f"Train % UP : {(train_df['price_pct_change'] > 0).mean()*100:.2f}%")
print(f"Val % UP   : {(val_df['price_pct_change'] > 0).mean()*100:.2f}%")
print(f"Test % UP  : {(test_df['price_pct_change'] > 0).mean()*100:.2f}%")

=== Prediction Magnitude ===
Actual    mean: 0.0157   std: 2.0782
Predicted mean: -0.2977   std: 0.2887

=== Directional Breakdown ===
Overall Dir Acc : 53.96%
When actual UP  : 16.67%  (192 samples)
When actual DOWN: 87.74%  (212 samples)

=== Prediction Bias ===
% predicted UP  : 14.36%
% actual UP     : 47.52%
=== Mean price_pct_change by split ===
Train mean : -0.0342%
Val mean   : -0.1905%
Test mean  : -0.0306%
Train % UP : 43.96%
Val % UP   : 42.24%
Test % UP  : 46.31%
